# Notebook 02 - Model Training and Comparison

This notebook trains and analyses five model families for the used-car price prediction project:

1. Ridge Regression
2. K-Nearest Neighbours Regression
3. Decision Tree Regression
4. Random Forest Regression
5. Multi-layer Perceptron Regression, used as the neural network model

The design follows the Assignment 2 requirements: models must be different algorithms, every algorithm must tune at least one hyperparameter, and all experiments must use the same validation approach. The test set is reserved for final evaluation by the evaluation notebook; this notebook reports train/dev results and exports predictions, metrics, fitted models, and split indices for the rest of the group.

**Research-question links**

- **Q1 Core predictive ability:** establish whether vehicle attributes can predict used-car prices.
- **Q2 Model comparison:** compare linear, instance-based, tree-based, ensemble, and neural network models.
- **Q3 Data preprocessing impact:** compare scaling strategies for models that are sensitive to feature scale.
- **Q4 Feature influence:** use Ridge coefficients and Random Forest feature importances as interpretable evidence.

## 1. Setup

In [ ]:
import json
import os
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import ParameterGrid, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", "{:.4f}".format)

%matplotlib inline
plt.style.use("default")

RANDOM_STATE = 49

# Use all CPU cores where sklearn supports it. If this causes issues in a restricted
# environment, fix the Jupyter/runtime permissions rather than changing this to 1.
N_JOBS = -1

# Set to "full" for final overnight/full-machine runs. The default keeps notebook
# iteration practical while final dev evaluation still uses the complete dev set.
TUNING_MODE = "balanced"  # options: "balanced", "full"

cwd = Path.cwd().resolve()
if (cwd / "code" / "data" / "processed" / "cars_cleaned.csv").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data" / "processed" / "cars_cleaned.csv").exists():
    PROJECT_ROOT = cwd.parents[1]
else:
    raise FileNotFoundError("Run this notebook from the project root or from code/notebooks.")

DATA_PATH = PROJECT_ROOT / "code" / "data" / "processed" / "cars_cleaned.csv"
OUTPUT_DIR = PROJECT_ROOT / "code" / "outputs" / "model_training_comparison"
MODEL_DIR = OUTPUT_DIR / "models"
FIG_DIR = OUTPUT_DIR / "figures"

for path in [OUTPUT_DIR, MODEL_DIR, FIG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data path: {DATA_PATH}")
print(f"Output directory: {OUTPUT_DIR}")

## 2. Load Data

Notebook 01 exports `cars_cleaned.csv`. It includes raw categorical variables and also target-encoded columns (`brand_mean_price`, `model_mean_price`). Those target-encoded columns were computed on the full dataset for EDA, so they are excluded here to avoid leakage. Instead, `brand`, `model`, `transmission`, and `fuelType` are encoded inside the model pipeline using the training split only.

In [ ]:
df = pd.read_csv(DATA_PATH)
df["row_id"] = df.index

print(f"Dataset shape: {df.shape}")
print(f"Missing values: {df.isna().sum().sum()}")
display(df.head())
display(df[["price", "log_price", "car_age", "mileage", "engineSize"]].describe())

## 3. Features, Target, and Shared Split

The target is `log_price` because Notebook 01 showed that price is right-skewed. Metrics are also reported after converting predictions back to pounds, which makes the error easier to interpret in the report.

In [ ]:
TARGET_COL = "log_price"

excluded_cols = [
    "price", "log_price", "row_id",
    "brand_mean_price", "model_mean_price",  # full-data target encodings from Notebook 01
    "fuel_Diesel", "fuel_Electric", "fuel_Hybrid", "fuel_Petrol",  # recreated in pipeline
    "transmission_enc",  # recreated from raw transmission in pipeline
]

feature_cols = [c for c in df.columns if c not in excluded_cols]
X = df[feature_cols].copy()
y = df[TARGET_COL].copy()

numeric_cols = X.select_dtypes(include=["int64", "float64", "bool"]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

X_train_full, X_test, y_train_full, y_test, idx_train_full, idx_test = train_test_split(
    X, y, df["row_id"], test_size=0.20, random_state=RANDOM_STATE
)

X_train, X_dev, y_train, y_dev, idx_train, idx_dev = train_test_split(
    X_train_full, y_train_full, idx_train_full, test_size=0.25, random_state=RANDOM_STATE
)

split_indices = {
    "random_state": RANDOM_STATE,
    "target": TARGET_COL,
    "train_idx": idx_train.astype(int).tolist(),
    "dev_idx": idx_dev.astype(int).tolist(),
    "test_idx": idx_test.astype(int).tolist(),
}
with open(OUTPUT_DIR / "split_indices.json", "w", encoding="utf-8") as f:
    json.dump(split_indices, f, indent=2)

print(f"Features ({len(feature_cols)}): {feature_cols}")
print(f"Numeric ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical ({len(categorical_cols)}): {categorical_cols}")
print(f"Train/dev/test sizes: {len(X_train):,} / {len(X_dev):,} / {len(X_test):,}")

## 4. Preprocessing and Evaluation Helpers

In [ ]:
def sample_frame(X_source, y_source, max_rows=None, random_state=RANDOM_STATE):
    if max_rows is None or len(X_source) <= max_rows:
        return X_source, y_source
    sample_idx = X_source.sample(n=max_rows, random_state=random_state).index
    return X_source.loc[sample_idx], y_source.loc[sample_idx]


if TUNING_MODE == "full":
    TUNE_TRAIN_LIMIT = None
    TUNE_DEV_LIMIT = None
else:
    TUNE_TRAIN_LIMIT = 25000
    TUNE_DEV_LIMIT = 6000

X_tune_train, y_tune_train = sample_frame(X_train, y_train, TUNE_TRAIN_LIMIT)
X_tune_dev, y_tune_dev = sample_frame(X_dev, y_dev, TUNE_DEV_LIMIT)

print(f"Tuning mode: {TUNING_MODE}")
print(f"Rows used for tuning: train={len(X_tune_train):,}, dev={len(X_tune_dev):,}")


def make_preprocessor(scaling="standard"):
    if scaling == "none":
        numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    elif scaling == "standard":
        numeric_steps = [("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
    elif scaling == "minmax":
        numeric_steps = [("imputer", SimpleImputer(strategy="median")), ("scaler", MinMaxScaler())]
    else:
        raise ValueError(f"Unknown scaling option: {scaling}")

    categorical_steps = [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
    ]

    return ColumnTransformer([
        ("num", Pipeline(numeric_steps), numeric_cols),
        ("cat", Pipeline(categorical_steps), categorical_cols),
    ])


def make_pipeline(model, scaling="standard"):
    return Pipeline([
        ("preprocess", make_preprocessor(scaling=scaling)),
        ("model", model),
    ])


def regression_metrics(y_true_log, y_pred_log):
    y_true_price = np.expm1(y_true_log)
    y_pred_price = np.expm1(y_pred_log)
    return {
        "MAE_log": mean_absolute_error(y_true_log, y_pred_log),
        "RMSE_log": np.sqrt(mean_squared_error(y_true_log, y_pred_log)),
        "R2_log": r2_score(y_true_log, y_pred_log),
        "MAE_price": mean_absolute_error(y_true_price, y_pred_price),
        "RMSE_price": np.sqrt(mean_squared_error(y_true_price, y_pred_price)),
        "R2_price": r2_score(y_true_price, y_pred_price),
    }


def evaluate_model(model_name, pipeline, X_eval, y_eval, eval_set):
    pred_log = pipeline.predict(X_eval)
    row = {"model": model_name, "eval_set": eval_set}
    row.update(regression_metrics(y_eval, pred_log))
    return row, pred_log


def tune_model(model_name, estimator_factory, param_grid, scaling="standard"):
    rows = []
    best_pipe = None
    best_score = np.inf
    best_params = None

    for params in ParameterGrid(param_grid):
        pipe = make_pipeline(estimator_factory(**params), scaling=scaling)
        pipe.fit(X_tune_train, y_tune_train)
        row, _ = evaluate_model(model_name, pipe, X_tune_dev, y_tune_dev, "dev_tune")
        row.update(params)
        row["scaling"] = scaling
        row["tune_train_rows"] = len(X_tune_train)
        row["tune_dev_rows"] = len(X_tune_dev)
        rows.append(row)
        if row["RMSE_log"] < best_score:
            best_score = row["RMSE_log"]
            best_pipe = pipe
            best_params = params

    tuning_results = pd.DataFrame(rows).sort_values("RMSE_log", ascending=True).reset_index(drop=True)

    # Refit the selected configuration on the full training split before dev evaluation/export.
    final_pipe = make_pipeline(estimator_factory(**best_params), scaling=scaling)
    final_pipe.fit(X_train, y_train)
    return tuning_results, final_pipe, best_params

## 5. Tune and Train Five Models

The chosen model families cover a spectrum from high-bias/low-variance to more flexible and lower-bias models, which is required by the assignment brief.

In [ ]:
model_specs = {
    "ridge_regression": {
        "factory": lambda alpha: Ridge(alpha=alpha, random_state=RANDOM_STATE),
        "grid": {"alpha": [0.1, 1.0, 10.0, 100.0]},
        "scaling": "standard",
    },
    "knn_regression": {
        "factory": lambda n_neighbors, weights: KNeighborsRegressor(
            n_neighbors=n_neighbors, weights=weights, n_jobs=N_JOBS
        ),
        "grid": {"n_neighbors": [5, 10, 20], "weights": ["uniform", "distance"]},
        "scaling": "standard",
    },
    "decision_tree": {
        "factory": lambda max_depth, min_samples_leaf: DecisionTreeRegressor(
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            random_state=RANDOM_STATE,
        ),
        "grid": {"max_depth": [8, 16, None], "min_samples_leaf": [1, 10, 50]},
        "scaling": "none",
    },
    "random_forest": {
        "factory": lambda n_estimators, max_depth, min_samples_leaf: RandomForestRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            random_state=RANDOM_STATE,
            n_jobs=N_JOBS,
        ),
        "grid": {"n_estimators": [100, 200], "max_depth": [16, None], "min_samples_leaf": [1, 10]},
        "scaling": "none",
    },
    "mlp_neural_network": {
        "factory": lambda hidden_layer_sizes, alpha, learning_rate_init: MLPRegressor(
            hidden_layer_sizes=hidden_layer_sizes,
            alpha=alpha,
            learning_rate_init=learning_rate_init,
            early_stopping=True,
            validation_fraction=0.15,
            max_iter=300,
            random_state=RANDOM_STATE,
        ),
        "grid": {
            "hidden_layer_sizes": [(64,), (128,), (64, 32)],
            "alpha": [0.0001, 0.001],
            "learning_rate_init": [0.001],
        },
        "scaling": "standard",
    },
}

tuning_tables = []
best_models = {}
best_params_rows = []

for model_name, spec in model_specs.items():
    print(f"\nTuning {model_name}...")
    tuning_results, final_pipe, best_params = tune_model(
        model_name=model_name,
        estimator_factory=spec["factory"],
        param_grid=spec["grid"],
        scaling=spec["scaling"],
    )
    tuning_tables.append(tuning_results)
    best_models[model_name] = final_pipe
    best_params_rows.append({"model": model_name, "scaling": spec["scaling"], **best_params})
    joblib.dump(final_pipe, MODEL_DIR / f"{model_name}.joblib")
    display(tuning_results.head())

tuning_all = pd.concat(tuning_tables, ignore_index=True)
best_params_df = pd.DataFrame(best_params_rows)

tuning_all.to_csv(OUTPUT_DIR / "hyperparameter_tuning_results.csv", index=False)
best_params_df.to_csv(OUTPUT_DIR / "best_hyperparameters.csv", index=False)

display(best_params_df)

## 6. Full Dev Evaluation and Prediction Export

All selected models are evaluated on the same complete dev set. The test set is not used here.

In [ ]:
metric_rows = []
prediction_frames = []

for model_name, pipe in best_models.items():
    train_row, _ = evaluate_model(model_name, pipe, X_train, y_train, "train")
    dev_row, dev_pred_log = evaluate_model(model_name, pipe, X_dev, y_dev, "dev")
    metric_rows.extend([train_row, dev_row])

    prediction_frames.append(pd.DataFrame({
        "row_id": idx_dev.astype(int).values,
        "model": model_name,
        "y_true_log": y_dev.values,
        "y_pred_log": dev_pred_log,
        "y_true_price": np.expm1(y_dev.values),
        "y_pred_price": np.expm1(dev_pred_log),
    }))

metrics = pd.DataFrame(metric_rows).sort_values(["eval_set", "RMSE_log"])
dev_predictions = pd.concat(prediction_frames, ignore_index=True)

metrics.to_csv(OUTPUT_DIR / "model_metrics_train_dev.csv", index=False)
dev_predictions.to_csv(OUTPUT_DIR / "dev_predictions.csv", index=False)

display(metrics)
display(dev_predictions.head())

## 7. Preprocessing Impact: Scaling Comparison

This experiment supports the preprocessing research question. Ridge and KNN are compared under no scaling, standard scaling, and min-max scaling. KNN is expected to be most sensitive because it relies directly on distances.

In [ ]:
scaling_rows = []

ridge_alpha = best_params_df.loc[best_params_df["model"] == "ridge_regression", "alpha"].iloc[0]
knn_params = best_params_df.loc[best_params_df["model"] == "knn_regression"].iloc[0].to_dict()

for scaling in ["none", "standard", "minmax"]:
    ridge_pipe = make_pipeline(Ridge(alpha=ridge_alpha, random_state=RANDOM_STATE), scaling=scaling)
    ridge_pipe.fit(X_train, y_train)
    row, _ = evaluate_model("ridge_regression", ridge_pipe, X_dev, y_dev, "dev")
    row["scaling"] = scaling
    scaling_rows.append(row)

    knn_pipe = make_pipeline(
        KNeighborsRegressor(
            n_neighbors=int(knn_params["n_neighbors"]),
            weights=knn_params["weights"],
            n_jobs=N_JOBS,
        ),
        scaling=scaling,
    )
    knn_pipe.fit(X_tune_train, y_tune_train)
    row, _ = evaluate_model("knn_regression", knn_pipe, X_tune_dev, y_tune_dev, "dev_tune")
    row["scaling"] = scaling
    row["note"] = "KNN scaling comparison uses tuning subset for runtime."
    scaling_rows.append(row)

scaling_results = pd.DataFrame(scaling_rows)
scaling_results.to_csv(OUTPUT_DIR / "scaling_comparison.csv", index=False)
display(scaling_results)

plot_data = scaling_results.pivot(index="scaling", columns="model", values="RMSE_price").loc[["none", "standard", "minmax"]]
ax = plot_data.plot(kind="bar", figsize=(9, 5))
ax.set_title("Figure 1: Scaling Impact on RMSE")
ax.set_xlabel("Scaling strategy")
ax.set_ylabel("RMSE on price scale")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIG_DIR / "scaling_impact_rmse.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Feature Influence

Ridge coefficients provide an interpretable linear view. Random Forest importances provide a non-linear tree-based view. These are not causal claims; they are model-based evidence for Q4.

In [ ]:
ridge_model = best_models["ridge_regression"]
feature_names = ridge_model.named_steps["preprocess"].get_feature_names_out()
ridge_coef = ridge_model.named_steps["model"].coef_

ridge_coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": ridge_coef,
})
ridge_coef_df["abs_coefficient"] = ridge_coef_df["coefficient"].abs()
ridge_coef_df = ridge_coef_df.sort_values("abs_coefficient", ascending=False).reset_index(drop=True)
ridge_coef_df.to_csv(OUTPUT_DIR / "ridge_feature_coefficients.csv", index=False)

rf_model = best_models["random_forest"]
rf_feature_names = rf_model.named_steps["preprocess"].get_feature_names_out()
rf_importances = rf_model.named_steps["model"].feature_importances_
rf_importance_df = pd.DataFrame({
    "feature": rf_feature_names,
    "importance": rf_importances,
}).sort_values("importance", ascending=False).reset_index(drop=True)
rf_importance_df.to_csv(OUTPUT_DIR / "random_forest_feature_importance.csv", index=False)

display(ridge_coef_df.head(15))
display(rf_importance_df.head(15))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ridge_plot = ridge_coef_df.head(12).iloc[::-1]
axes[0].barh(ridge_plot["feature"], ridge_plot["coefficient"])
axes[0].set_title("Figure 2a: Top Ridge Coefficients")
axes[0].set_xlabel("Coefficient for log_price")

rf_plot = rf_importance_df.head(12).iloc[::-1]
axes[1].barh(rf_plot["feature"], rf_plot["importance"])
axes[1].set_title("Figure 2b: Top Random Forest Importances")
axes[1].set_xlabel("Feature importance")

plt.tight_layout()
plt.savefig(FIG_DIR / "feature_influence.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Handoff Files

The following outputs are intended for the evaluation and report-integration notebooks:

- `split_indices.json`: fixed train/dev/test rows.
- `model_metrics_train_dev.csv`: train and dev metrics for all five models.
- `dev_predictions.csv`: row-level dev predictions for residual and error analysis.
- `hyperparameter_tuning_results.csv`: all tuning attempts and metrics.
- `best_hyperparameters.csv`: selected model settings.
- `ridge_feature_coefficients.csv` and `random_forest_feature_importance.csv`: feature influence evidence.
- `models/*.joblib`: fitted pipelines that include preprocessing and model steps.

In [ ]:
print("Saved outputs:")
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(PROJECT_ROOT))